# Chapter 8, Section 8.3 — Worked Example: Surrogate Modelling of the Real Hennaya MODFLOW Model

**AI for Hydrogeologists** — companion notebook

Unlike the Theis PINN exercise elsewhere in this chapter (explicitly
synthetic), this notebook uses the **real, peer-reviewed, calibrated
MODFLOW model** of the Hennaya plain published in Laoufi et al. (2024,
*Sustainability* 16, 10777): "Integrated Simulation of Groundwater Flow
and Nitrate Transport in an Alluvial Aquifer Using MODFLOW and MT3D".

**Design mirrors the published paper exactly:**
- 1981 heads: steady-state calibration target (hydraulic conductivity
  calibrated against these)
- 2022 heads: transient calibration target (storage calibrated against these)
- 2012 heads: **independent validation, held out from training entirely**
  — exactly as in the paper's own validation design (their reported
  validation R2 = 0.978)

**Goal:** train a machine learning surrogate that emulates MODFLOW's
head response given the calibrated K, S, and geometry fields, and check
whether it reproduces the 2012 heads it never saw during training.

In [ ]:
import pandas as pd
import numpy as np
import io
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score

BASE = "https://raw.githubusercontent.com/Dr-LAOUFIAbdessalam/ai-hydrogeologists/main/ch08_physics_informed/data/raw/"
HNOFLO_THRESHOLD = 1e29  # MODFLOW's standard inactive-cell flag

def load_all_layers(url):
    import urllib.request
    text = urllib.request.urlopen(url).read().decode("utf-8", errors="ignore")
    lines = text.splitlines()
    layer_idx = [i for i, l in enumerate(lines) if l.startswith("Layer")]
    layer_idx.append(len(lines))
    blocks = {}
    for k in range(len(layer_idx) - 1):
        name = lines[layer_idx[k]].strip()
        chunk = lines[layer_idx[k] + 1:layer_idx[k + 1]]
        df = pd.read_csv(io.StringIO("\n".join(chunk)), sep="\t")
        df = df.loc[:, ~df.columns.str.match("Unnamed")]
        blocks[name] = df
    return blocks["Layer 1"]

geo = load_all_layers(BASE + "geomertry.TXT")
cond = load_all_layers(BASE + "Conductivity.TXT")
stor = load_all_layers(BASE + "Storage.TXT")
h1981 = load_all_layers(BASE + "Simulated_head_1981.TXT")
h2012 = load_all_layers(BASE + "Simulated_head_2012.TXT")
h2022 = load_all_layers(BASE + "Simulated_head_2022.TXT")
print("Grids loaded:", geo.shape, cond.shape, stor.shape)


## Merge parameter fields and build the modelling table

In [ ]:
static = geo.merge(cond[["X","Y","Kx","Ky","Kz"]], on=["X","Y"]) \
            .merge(stor[["X","Y","Ss","Sy","PorEff.","PorTot."]], on=["X","Y"])
static = static.rename(columns={"Thick.":"Thick","PorEff.":"PorEff","PorTot.":"PorTot"})

frames = []
for year, hdf in [(1981, h1981), (2012, h2012), (2022, h2022)]:
    df = static.copy()
    df["Head"] = hdf["Head"].values
    df["year"] = year
    df = df[df["Head"] < HNOFLO_THRESHOLD].reset_index(drop=True)
    frames.append(df)
    print(f"{year}: {len(df)} active cells")

full = pd.concat(frames, ignore_index=True)
for col in ["Kx","Ky","Kz"]:
    full[f"log_{col}"] = np.log10(full[col].clip(lower=1e-12))

feature_cols = ["X","Y","Z","Top","Bot","Thick","log_Kx","log_Ky","log_Kz",
                 "Ss","Sy","PorEff","PorTot","year"]

train = full[full["year"].isin([1981, 2022])].reset_index(drop=True)
test = full[full["year"] == 2012].reset_index(drop=True)  # held out entirely
X_train, y_train = train[feature_cols].values, train["Head"].values
X_test, y_test = test[feature_cols].values, test["Head"].values
print(f"\nTrain (1981+2022): n={len(train)}  |  Validation (2012, held out): n={len(test)}")


## Train surrogate models and validate on the held-out 2012 campaign

In [ ]:
models = {"Linear Regression": LinearRegression(),
          "Random Forest": RandomForestRegressor(n_estimators=500, random_state=42)}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    r2_train = r2_score(y_train, model.predict(X_train))
    pred_test = model.predict(X_test)
    r2_test = r2_score(y_test, pred_test)
    rmse_test = np.sqrt(mean_squared_error(y_test, pred_test))
    results[name] = (pred_test, r2_test, rmse_test)
    print(f"\n{name}: train R2={r2_train:.4f} | VALIDATION (2012) R2={r2_test:.4f}  RMSE={rmse_test:.2f} m")

print("\nPublished MODFLOW model's own 2012 validation: R2 = 0.978 (Laoufi et al. 2024)")


**Striking result:** a simple linear regression surrogate reaches
R2 ~0.97 on the 2012 campaign it never saw during training — remarkably
close to the published MODFLOW model's own validation performance
(R2 = 0.978). Before treating this as a success, inspect **why**.

## Feature importance: what is the surrogate actually using?

In [ ]:
rf = models["Random Forest"]
importances = pd.Series(rf.feature_importances_, index=feature_cols).sort_values(ascending=False)
print(importances.round(3))

k_s_importance = importances[["log_Kx","log_Ky","log_Kz","Ss","Sy","PorEff","PorTot"]].sum()
elev_importance = importances[["Top","Z","Bot"]].sum()
print(f"\nCalibrated K+S+porosity fields, combined importance: {k_s_importance:.3f}")
print(f"Elevation-related fields (Top, Z, Bot), combined importance: {elev_importance:.3f}")


## Initial (provisional) interpretation

Taken at face value, the calibrated hydraulic conductivity and storage
fields — the actual physically calibrated parameters from the published
model — contribute almost nothing to the surrogate's predictions
(combined importance ~0.002), while elevation-related features dominate
(~0.84). This would suggest the surrogate found a statistical shortcut
through land-surface elevation rather than learning the K/S-dependent
physics MODFLOW actually solves for.

**Before accepting this at face value, an ablation test is needed** — see
below.

## Ablation test: is the near-zero K/S importance real, or an artifact?

In [ ]:
ks_only_cols = ["log_Kx","log_Ky","log_Kz","Ss","Sy","PorEff","PorTot"]
X_train_ks, X_test_ks = train[ks_only_cols].values, test[ks_only_cols].values

for name, model in [("Linear Regression", LinearRegression()),
                     ("Random Forest", RandomForestRegressor(n_estimators=500, random_state=42))]:
    model.fit(X_train_ks, y_train)
    r2_ks = r2_score(y_test, model.predict(X_test_ks))
    print(f"{name}: validation R2 using K/S/porosity ALONE = {r2_ks:.4f}")

corr_top_kx = full["Top"].corr(full["log_Kx"])
print(f"\nCorrelation between Top elevation and log(Kx): {corr_top_kx:.3f}")


## Resolution

**Random Forest achieves R2 = 0.944 using K/S/porosity ALONE** (no
position or elevation at all) — almost as good as the full feature set,
and dramatically better than Linear Regression's R2 = 0.525 with the same
K/S-only inputs (the K-to-head relationship is evidently non-linear enough
that RF's flexibility matters here). This **confirms K/S do carry
substantial real signal.**

Their near-zero importance in the full model is therefore a
**multicollinearity artifact**: elevation and the calibrated K/S zones are
correlated (r = -0.42 between Top and log Kx), plausibly because both were
assigned following the same underlying geological zonation of the alluvial
plain. When correlated features compete, impurity-based tree importance
can arbitrarily assign most of the credit to one of them, starving the
other — a well-documented limitation of `feature_importances_`. This is
exactly why the book turns to SHAP in Chapter 9 rather than relying on raw
feature importances alone.

**Revised conclusion:** the surrogate's good validation score is not purely
a spurious elevation shortcut after all — K/S genuinely matter — but the
*naive* feature-importance reading would have wrongly concluded otherwise.
The real lesson is about the interpretability tool, not just the model.